# Real Data Performance
### Predict perfromance of real 3E cells using machine learning. This code only trains the GP model.
### To compare with the ensemble model, save your predictions yPred_gp and reload them in the Model_Perofrmance_3_10 notebook.

In [3]:
pip install tensorflow-probability==0.20.1

   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/6.9 MB 3.2 MB/s eta 0:00:03
   -- ------------------------------------- 0.4/6.9 MB 4.5 MB/s eta 0:00:02
   ---- ----------------------------------- 0.7/6.9 MB 5.5 MB/s eta 0:00:02
   ----- ---------------------------------- 0.9/6.9 MB 5.4 MB/s eta 0:00:02
   ------ --------------------------------- 1.2/6.9 MB 5.5 MB/s eta 0:00:02
   -------- ------------------------------- 1.4/6.9 MB 5.4 MB/s eta 0:00:02
   --------- ------------------------------ 1.7/6.9 MB 5.2 MB/s eta 0:00:02
   ----------- ---------------------------- 2.0/6.9 MB 5.3 MB/s eta 0:00:01
   ------------ --------------------------- 2.1/6.9 MB 5.0 MB/s eta 0:00:01
   ------------- -------------------------- 2.3/6.9 MB 4.9 MB/s eta 0:00:01
   -------------- ------------------------- 2.5/6.9 MB 4.8 MB/s eta 0:00:01
   -------------- ------------------------- 2.5/6.9 MB 4.5 MB/s eta 0:00:01
   --------------- 

In [1]:
#import libraries

# import pybamm 

import tensorflow as tf
import gpflow

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib qt
import itertools
import os
import datetime

from scipy import integrate
import scipy.stats
import pickle
import mgzip
import pandas as pd
import importlib

from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d
import math

from sklearn.metrics import mean_squared_error


# from statsmodels.graphics.tsaplots import plot_acf

import time


from sklearn.preprocessing import RobustScaler

import subprocess
from subprocess import call

# from xlsx2csv import Xlsx2csv


In [2]:
#parameters

import datetime
date = datetime.date.today().strftime('%Y-%m-%d')
date = "2024-04-15"
dataset_filename = "data110323"

In [3]:
#reload data
xCC = pd.read_pickle(r'Features/' + date + r'/xCC.pkl')
yCC = pd.read_pickle(r'Features/' + date + r'/yCC.pkl')

#xCC = pd.read_pickle(r'Features/' + "2023-10-09" + r'/xCC.pkl')
#yCC = pd.read_pickle(r'Features/' + "2023-10-09" + r'/yCC.pkl')



In [4]:
with open(r'Features/' + date + r'/masterList.pkl', 'rb') as f:
    masterList = pickle.load(f)


totalCells = max(masterList.keys())
masterList

cell_num
1        0.33
2        0.33
3        0.33
4        0.33
5         0.5
6         0.5
7         0.5
8         0.5
9         0.5
10        0.8
11        0.8
12        0.8
13        0.8
14        0.8
15        0.8
16        0.8
17        0.8
18        0.8
19        0.8
20        0.8
21        0.8
22        0.8
23        0.8
24        0.8
25        0.8
26        0.8
27        0.8
28        0.8
29          1
30          1
31          1
32          1
33          1
34          1
35          1
36          1
37          1
38          1
39          1
40          1
41          1
42          1
43          1
44          1
45          1
46          1
47          1
48          1
49          1
50    stepped
51    stepped
52    stepped
53    stepped
54    stepped
55    stepped
Name: C-rate, dtype: object

In [5]:
from functions import TrainTestSplit
importlib.reload(TrainTestSplit)

#testing 4/19, 3.55 activation
# cell_test = [3,5, 9,12]

#testing 4/19, Canrd Li
# cell_test = [3,5,7]

#testing 4/19, all stepped in test
# cell_test = [11,12]

# cell_test = [3,7, 12, 15, 20]

cell_test = [3,8,15,19,30,31,34,51]

cell_train = np.array([x for x in masterList.keys() if x not in cell_test])

xTrain, yTrain, xTest, yTest = TrainTestSplit.train_test_split(xCC, yCC, 'cell', cell_test, masterList)
timeTrain = xTrain['Time']
timeTest = xTest['Time']

#5


print(xTrain.shape, xTest.shape)
print(len(xTrain)/(len(xCC)))

Updated
Training list:  [1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 32, 33, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 55]
Testing list:  [3, 8, 15, 19, 30, 31, 34, 51]
(7523137, 65) (2162149, 65)
0.7767594059690132


In [6]:
masterList[[val for val in cell_test]]

cell_num
3        0.33
8         0.5
10        0.8
14        0.8
16        0.8
24          1
31          1
32          1
39    stepped
Name: C-rate, dtype: object

In [12]:
for cell_num in masterList.keys():
    idx = [xCC.index.get_level_values(0) == cell_num][0]
    print(cell_num, len(xCC[idx]))

1 817123
2 671602
3 814892
4 556644
5 590983
6 153715
7 442828
8 558026
9 390522
10 287534
11 97347
12 286414
13 174056
14 179606
15 94412
16 145957
17 75401
18 11782
19 167791
20 152300
21 63936
22 61210
23 192421
24 152864
25 117360
26 96585
27 58231
28 100956
29 77875
30 193803
31 77576
32 69214
33 17361
34 90883
35 122494
36 14775
37 6118
38 90904
39 204472
40 42697
41 51930
42 14363
43 54378
44 26378
45 5445
46 148463
47 28234
48 29472
49 1255
50 1367
51 315331
52 169854
53 18918
54 22168
55 233782
56 77051


### Data Visualization
### Compare some of the features created between training and testing sets. It is expected that features with good overlap will be better predictors overall.

In [ ]:

cols = ['Voltage', 'Energy', 'Time', 'Overpotential', 'Capacity', 'Overpotential*Cap_overpotential', 'Cap_overpotential', 'CE', 'CE*Capacity']
cols = ['Voltage', 'Energy', 'Capacity', 'CE*Capacity', 'VE', 'VE*Capacity', 'VE*Voltage', 'EE', 'EE*Capacity']

cols = ['Voltage', 'Energy', 'Capacity', 'Power', 'Overpotential', 'Time']

# cols = ['Power', 'Voltage-8', 'Power-8', 'Capacity*Overpotential-8', 'Cycle*Voltage-8']
# cols = ['Capacity-8', 'Energy-8', 'Capacity*Cap_overpotential-8', 'Time-6',  'Energy-6']
# cols = ['Capacity-8', 'Voltage-8', 'Energy-8', 'Cap_overpotential-8', 'Energy', 'Time']
cols = use_cols
from functions import DataPlotting
importlib.reload(DataPlotting)
# cell_test = [4]

# temp = [1,2,4,5,6,7]
DataPlotting.plot_one_cycle_multiple_cells(xCC, yCC, 4, cell_train, cell_test, cols)

In [ ]:
cols = ['Voltage', 'Energy', 'Capacity', 'Capacity*Overpotential', 'Overpotential', 'Time',]
from functions import DataPlotting
importlib.reload(DataPlotting)
c_rate = 0.5
select_cells = masterList[masterList.values == str(c_rate)]
DataPlotting.plot_one_cycle_multiple_cells(xCC, yCC, 1, select_cells.keys(), [], cols)
plt.suptitle('Cycle 3, ' + str(c_rate)  + 'C')

Text(0.5, 0.98, 'Cycle 3, 0.5C')

## Feature Selection

In [6]:
input_scaler = RobustScaler()

xTrain_scaled = xTrain.loc[::8,:].copy()

xTrain_scaled.values[:]  = input_scaler.fit_transform(xTrain.loc[::8,:])



In [ ]:
# from sklearn.svm import SVR
from sklearn.feature_selection import RFE

# Create the RFE object and rank each pixel
rfr = RandomForestRegressor(n_estimators=30, n_jobs = 6 )
rfe = RFE(estimator=rfr, n_features_to_select=5, step=15,verbose = 2)
rfe.fit(xTrain_scaled, yTrain.loc[::8])
ranking = rfe.ranking_


Fitting estimator with 105 features.
Fitting estimator with 90 features.
Fitting estimator with 75 features.
Fitting estimator with 60 features.
Fitting estimator with 45 features.
Fitting estimator with 30 features.
Fitting estimator with 15 features.


In [ ]:
#names of selected features
xTrain.iloc[:, rfe.ranking_ == 1].columns

Index(['Throughput', 'Power', 'Cycle*Capacity', 'Power-8', 'Cycle*Voltage-8'], dtype='object')

In [ ]:
#obtain feature importance values
forest_regr = RandomForestRegressor(n_estimators=30, n_jobs=8, verbose =2 )
forest_regr.fit( xTrain_scaled.iloc[:,rfe.ranking_ == 1], yTrain.loc[::8]) 
print(forest_regr.feature_importances_)
xTrain.loc[:,rfe.ranking_ == 1].columns

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.


building tree 1 of 30
building tree 2 of 30
building tree 3 of 30
building tree 4 of 30
building tree 5 of 30
building tree 6 of 30
building tree 7 of 30
building tree 8 of 30
building tree 9 of 30
building tree 10 of 30
building tree 11 of 30
building tree 12 of 30
building tree 13 of 30
building tree 14 of 30
building tree 15 of 30
building tree 16 of 30
building tree 17 of 30
building tree 18 of 30
building tree 19 of 30
building tree 20 of 30
building tree 21 of 30
building tree 22 of 30
building tree 23 of 30
building tree 24 of 30
building tree 25 of 30
building tree 26 of 30
building tree 27 of 30
building tree 28 of 30
building tree 29 of 30
building tree 30 of 30


[Parallel(n_jobs=8)]: Done  30 out of  30 | elapsed:  1.0min finished


[0.04735598 0.06059244 0.66987886 0.08693195 0.13524077]


Index(['Voltage', 'Energy', 'Power', 'Current*Overpotential', 'Power-8'], dtype='object')

#### Now that we know the important features, we can drop columns to obtain only the useful ones.

In [6]:

# use_cols = xTrain.iloc[:, rfe.ranking_ == 1].columns

use_cols = ['Energy', 'Capacity', 'Voltage-8', 'Energy-8', 'Power-8']

xTrain = xTrain[use_cols]
xTest = xTest[use_cols]

## Model Development

In [7]:
#Scale the data. Required for GPR
input_scaler = RobustScaler()

xTrain_scaled = xTrain.copy()
xTrain_minmax = input_scaler.fit_transform(xTrain)
xTrain_scaled.values[:]  = xTrain_minmax
xTest_minmax = input_scaler.transform(xTest)

xTrain_minmax.shape



#xTrain_minmax = xTrain
#xTest_minmax = xTest

(7523137, 5)

### GP Model

In [8]:
#model setup

inputDim =xTrain_minmax.shape[1]
outputDim = 1

var_val = np.sqrt(0.002**2 / len(xTrain))

k = gpflow.kernels.Matern52(lengthscales= np.ones(inputDim)) + gpflow.kernels.Constant() + gpflow.kernels.White(variance=var_val)

#num_points = 800

num_points = 800

base = 15
inc = base * round(np.ceil(xTrain_minmax.shape[0]/num_points)/base)

Z = xTrain_minmax[:num_points*base:base, :].copy()

m = gpflow.models.SVGP(kernel=k, likelihood= gpflow.likelihoods.Gaussian(),inducing_variable=Z, num_data = xTrain_minmax.shape[0])


data = (xTrain_minmax, yTrain)

elbo = tf.function(m.elbo)



In [9]:

print("GPflow version:", gpflow.__version__)

GPflow version: 2.5.2


In [9]:
#model training

from gpflow.ci_utils import ci_niter

# from gpflow.ci_utils import reduce_in_tests
import gpflow.optimizers

def run_adam(model, iterations):
    """
    Utility function running the Adam optimizer

    :param model: GPflow model
    :param interations: number of iterations
    """
    # Create an Adam Optimizer action
    logf = []
    train_iter = iter(train_dataset.batch(minibatch_size))
    training_loss = model.training_loss_closure(train_iter, compile=True)
    optimizer = tf.optimizers.Adam(learning_rate=0.001)

    @tf.function
    def optimization_step():
        optimizer.minimize(training_loss, model.trainable_variables)

    for step in range(iterations):
        optimization_step()
        if step % 10 == 0:
            elbo = -training_loss().numpy()
            logf.append(elbo)
    return logf


    # @tf.function
    # def optimization_step(x_batch, y_batch):
    #     with tf.GradientTape() as tape:
    #         loss = model.training_loss_closure((x_batch, y_batch), compile=True)
    #     if tf.is_tensor(loss):  # Ensure loss is a tensor
    #         gradients = tape.gradient(loss, model.trainable_variables)
    #         optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    #     else:
    #         print("Loss is not a tensor:", loss)
    #         print("Model training_loss_closure:", model.training_loss_closure)
    #         print("x_batch shape:", x_batch.shape)
    #         print("y_batch shape:", y_batch.shape)

    # # Run optimization loop
    # logf = []
    # for step in range(iterations):
    #     for x_batch, y_batch in train_dataset.batch(minibatch_size):
    #         optimization_step(x_batch, y_batch)
    #     if step % 10 == 0:
    #         elbo = -model.training_loss().numpy()
    #         logf.append(elbo)
    # return logf



train_dataset = tf.data.Dataset.from_tensor_slices((xTrain_minmax, pd.DataFrame(yTrain).to_numpy())).repeat().shuffle(len(yTrain))

#minibatch_size = 256

#minibatch_size = 128

minibatch_size = 32


#maxiter = ci_niter(25000)
maxiter = ci_niter(5000)

logf = run_adam(m, maxiter)

params = gpflow.utilities.parameter_dict(m)

#save data
if not os.path.exists('models/' + date + '/' + dataset_filename):
    os.makedirs('models/' + date + '/' + dataset_filename)

with open('models/' + date + '/' + dataset_filename + '/svgp_model.pkl',"wb") as f:
    pickle.dump(params,f)



In [10]:
#check training metric. ELBO should increase then stabilize

plt.figure()
plt.plot(np.arange(maxiter)[::10], logf)
plt.xlabel('Number of iterations')
plt.ylabel('ELBO')


Text(0, 0.5, 'ELBO')

In [11]:
#make predictions

val = len(xTest_minmax)/10

yPred_gp = [] 
yStd = []
for i in range(10):
    start = int(val*i)
    end = int(val*(i+1))
    yPred_temp, yStd_temp = m.predict_f(xTest_minmax[start:end,:])
    yPred_gp.append(yPred_temp)
    yStd.append(yStd_temp)
yPred_gp = np.array(np.concatenate(yPred_gp)).flatten()
yExp = yTest[::].to_numpy().flatten()
yStd = 1.96 * np.sqrt(np.concatenate(yStd))

yPred = yPred_gp

np.savetxt('models/' + date + '/' + dataset_filename + 'svgp_results.npy', yPred_gp)

### Reload model. make sure you have set up the model itself.

In [ ]:
try:
    with open('models/' + date + '/' + dataset_filename + '/svgp_model.pkl', 'rb') as f:
        params = pickle.load(f)

    gpflow.utilities.multiple_assign(m, params)
except Exception as e:
    print(e)
    
yPred_gp = np.load('models/' + date + '/' + dataset_filename + 'svgp_results.npy')

### Baseline linear regression, if you want to compare in this notebook.

In [ ]:
from sklearn.linear_model import LinearRegression

baseline = LinearRegression()
reg = baseline.fit(xTrain_minmax, yTrain)
print(reg.coef_)
yPred_baseline = reg.predict(xTest_minmax)
# pickle.dumps(baseline)



filename = open('models/' + date + '/' + dataset_filename + '/baseline.pkl', 'wb')
pickle.dump(reg, filename)
filename.close()

[-2.81031503e-01  2.66525291e-01  2.81957351e-01 -3.16397857e-01
  1.83391353e-04]


## Evaluate Model Performance

In [ ]:
##linear regression for observing relationship between predicted and actual data

slope, intercept, r, p, stderr = scipy.stats.linregress(yPred.T, yExp.T)
line = f'Regression line: y={intercept:.2f}+{slope:.2f}x, r={r:.2f}'

plt.figure()
plt.plot(yPred, yExp, '*', label='data', markersize = 0.1)
plt.plot(yPred, intercept + slope * yPred, label=line)

plt.legend()
plt.xlabel('Predicted NEP')
plt.ylabel('Actual NEP')
#ax1.scatter(chargeTimeList*60, timeToPlating)
#ax1.set(xlabel='charge time [seconds]', ylabel= 'Time to plating [seconds]')

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 15)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 15)

In [ ]:

#specify which model to evaluate metrics for
yPred = yPred_gp

In [ ]:
from sklearn.metrics import mean_squared_error

#Get error metrics
#* 100 / len (yPred)
mpe = abs(yPred - yExp)
mae = mpe.mean()

rmse = mean_squared_error(yExp, yPred, squared=False)
rmspe = np.sqrt(np.mean(np.square(((yExp - yPred) / abs(yExp))), axis=0)) * 100

print('MAE:', "{:.5f}".format(mae), 'V')
# print('RMSPE:', "{:.5f}".format
# (rmspe), '%')
print('RMSE:', "{:.5f}".format(rmse), 'V')
##plot absolute differences


plt.figure()
plt.plot(yExp, abs(yExp - yPred), 'r*',markersize = 0.7)

# plt.legend()
plt.xlabel('Actual NEP', fontsize=12)
plt.ylabel('Absolute NEP Difference', fontsize=12)
#ax1.scatter(chargeTimeList*60, timeToPlating)
#ax1.set(x  label='charge time [seconds]', ylabel= 'Time to plating [seconds]')

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 12)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 12)

MAE: 0.01553 V
RMSE: 0.02018 V


In [ ]:
yPred = yPred_baseline

In [ ]:
from sklearn.metrics import mean_squared_error

#Get error metrics
#* 100 / len (yPred)
mpe = abs(yPred - yExp)
mae = mpe.mean()

rmse = mean_squared_error(yExp, yPred, squared=False)
rmspe = np.sqrt(np.mean(np.square(((yExp - yPred) / abs(yExp))), axis=0)) * 100

print('MAE:', "{:.5f}".format(mae), 'V')
# print('RMSPE:', "{:.5f}".format
# (rmspe), '%')
print('RMSE:', "{:.5f}".format(rmse), 'V')
##plot absolute differences


plt.figure()
plt.plot(yExp, abs(yExp - yPred), 'r*',markersize = 0.7)

# plt.legend()
plt.xlabel('Actual NEP', fontsize=12)
plt.ylabel('Absolute NEP Difference', fontsize=12)
#ax1.scatter(chargeTimeList*60, timeToPlating)
#ax1.set(x  label='charge time [seconds]', ylabel= 'Time to plating [seconds]')

ax = plt.gca()
ax.tick_params(axis = 'both', which = 'major', labelsize = 12)
ax.tick_params(axis = 'both', which = 'minor', labelsize = 12)

print('Averages across cells')

mae_list = []
rmse_list = []
for cell_num in cell_test:
    idx = [xTest.index.get_level_values(0) == cell_num][0]
    mae = abs(yPred[idx] - yExp[idx]).mean()
    rmse = mean_squared_error(yExp[idx], yPred[idx], squared=False)
    print('Dataset size: ', len(yPred[idx]) )
    print('MAE:', "{:.5f}".format(mae), 'V')
    print('RMSE:', "{:.5f}".format(rmse), 'V')



MAE: 0.01418 V
RMSE: 0.01928 V
Averages across cells
Dataset size:  519811
MAE: 0.00731 V
RMSE: 0.00949 V
Dataset size:  442995
MAE: 0.02016 V
RMSE: 0.02483 V
Dataset size:  44364
MAE: 0.01883 V
RMSE: 0.02451 V
Dataset size:  82301
MAE: 0.01850 V
RMSE: 0.02341 V
Dataset size:  69000
MAE: 0.01367 V
RMSE: 0.01888 V
Dataset size:  34876
MAE: 0.01766 V
RMSE: 0.02142 V
Dataset size:  20160
MAE: 0.01215 V
RMSE: 0.01544 V
Dataset size:  71729
MAE: 0.01864 V
RMSE: 0.02383 V


In [ ]:
##linear regression comparison
from scipy.stats import pearsonr
#  'gp': yPred_gp
yPred_list = {'baseline': yPred_baseline, 'gp': yPred_gp }
select_cell = 4
# cycle = np.arange(0, 10, 1)

for select_cell in range(len(cell_test)):
    indList = timeTest.index.get_loc(tuple([cell_test[select_cell]]))
    for key in yPred_list:
        plt.figure(figsize=(7.04, 5.28))
        corr, _ = pearsonr(yExp[indList], yPred_list[key][indList])
        corr *=100
        plt.plot(yExp[indList], yPred_list[key][indList], '*', label='data, r^2 = %.2f' % corr, markersize = 0.7)
        plt.plot(yExp[indList], yExp[indList])
        
        plt.legend()
        plt.xlabel('Observed NEP')
        plt.ylabel('Predicted NEP')
        ax = plt.gca()
        # ax.tick_params(axis = 'both', which = 'major', labelsize = 15)
        # ax.tick_params(axis = 'both', which = 'minor')
        # break
        
        plt.suptitle('Cell ' + str(cell_test[select_cell]) + ' ' + key)
        plt.savefig('models/' + date + '/' + dataset_filename + '/linear_regression_' + key + '__cell_' + str(select_cell))
        

In [ ]:
#error of individual cycles

plt.figure()

yPred_list = {'baseline': yPred_baseline, 'gp': yPred_gp }
#  'ensemble': yPred_ensemble,'gp': yPred_gp
cycle_error = {'baseline': [],'gp': []}
# ,'gp': []
# yPred_list = {'baseline': yPred_baseline}
# cycle_error = {'baseline': []}
select_cell = 0
# xTest.index.get_level_values(0) == 9

maxCycles = xTest.loc[cell_test[select_cell]].index.max()

for key, value in yPred_list.items():
    for cycle in range(1, maxCycles, 1):
        try:
            indList = timeTest.index.get_loc(tuple([cell_test[select_cell], cycle])) 
            cycle_error[key].append(mean_squared_error(yExp[indList], value[indList], squared=False))
        except:
            # print(key)
            cycle_error[key].append(np.nan)
    plt.plot(np.arange(1, maxCycles, 1),  cycle_error[key], '*-', linewidth = 2, label = key)


ax = plt.gca()
ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(nbins = 10, integer=True))
plt.xlabel('Cycle')
plt.ylabel('RMSE [V]')
plt.legend(prop = {'size':10})
plt.savefig('models/' + date + '/' + dataset_filename + '/individual_cycle_error_' + str(select_cell))

In [ ]:
cycle = 20

plt.rcParams['axes.labelsize'] = 13

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# cell_num = cell_test
select_cell = 0
indList = timeTest.index.get_loc(tuple([cell_test[select_cell], cycle])) 
# indList_lstm = timeTest.index.get_loc(tuple([cell_test, i]))

axs[0].plot(timeTest.iloc[indList], yPred_baseline[indList], linewidth = 2, label = 'baseline, Total RMSE = ' + "{:.4f}".format(mean_squared_error(yExp, yPred_baseline, squared=False)))
# axs[0].plot(timeTest.iloc[indList], yPred_ensemble[indList], linewidth = 2, label = 'ensemble, Total RMSE = ' + "{:.4f}".format(mean_squared_error(yExp, yPred_ensemble, squared=False)))
axs[0].plot(timeTest.iloc[indList], yPred_gp[indList], linewidth = 2, label = 'SVGP, Total RMSE = ' + "{:.4f}".format(mean_squared_error(yExp, yPred_gp, squared=False)))
axs[0].plot(timeTest.iloc[indList], yExp[indList], linewidth = 2, label = 'actual')

# axs[0].set(xlabel =  'Times [s]', ylabel = 'Anode Potential [V]', yticks = [0, 0.1, 0.2, 0.3, 0.4, 0.5] )
axs[0].set(xlabel =  'Times [s]', ylabel = 'Anode Potential [V]', yticks = [-0.05, 0, 0.05, 0.1] )
axs[0].legend(prop={'size':10})

axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_baseline[indList]), linewidth = 1, label = 'baseline')
# axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_ensemble[indList]), linewidth = 1, label = 'ensemble')
axs[1].plot(timeTest.iloc[indList], abs(yExp[indList] - yPred_gp[indList]), linewidth = 1, label = 'SVGP')




axs[1].set(xlabel =  'Times [s]', ylabel = 'Absolute V Difference'  )



fig.suptitle('Anode Potential Predictions for Cycle ' + str(cycle) ) 


print(abs(yPred_baseline - yExp).mean())
# print(abs(yPred_ensemble - yExp).mean())
print(abs(yPred_gp - yExp).mean())

plt.savefig('models/' + date + '/' + dataset_filename + '/one_cycle_error_comparison__cell_' + str(select_cell) + '_cycle_' + str(cycle))


0.016293772697402106
0.01327123883261529
